In [1]:
import pandas as pd
import numpy as np

In [2]:
news_df = pd.read_csv("analyst_ratings_processed.csv")

print(news_df.shape)
news_df.head()

(1400469, 4)


,Unnamed: 0,title,date,stock
0,0.0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A
1,1.0,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A
2,2.0,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A
3,3.0,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A
4,4.0,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A


In [3]:
news_df = news_df[["title", "date", "stock"]]

In [4]:
news_df.rename(
    columns={
        "title": "Headline",
        "date": "Timestamp",
        "stock": "Ticker"
    },
    inplace=True
)

In [5]:
stocks = ["AAPL", "AMZN", "GOOGL", "NVDA"]

news_df = news_df[
    news_df["Ticker"].isin(stocks)
].copy()

In [6]:
news_df.drop_duplicates(
    subset=["Headline", "Timestamp", "Ticker"],
    inplace=True
)

news_df.reset_index(drop=True, inplace=True)

In [7]:
news_df["Timestamp"] = pd.to_datetime(
    news_df["Timestamp"],
    utc=True
)

In [8]:
news_df["Timestamp"] = (
    news_df["Timestamp"]
    .dt.tz_convert("America/New_York")
)

In [11]:
from pandas.tseries.offsets import BDay

# Normalize timestamp to midnight while keeping timezone
news_df["Date"] = news_df["Timestamp"].dt.normalize()

# News published at or after 4:00 PM ET is assigned to the next business day
after_close = news_df["Timestamp"].dt.hour >= 16

news_df.loc[after_close, "Date"] = (
    news_df.loc[after_close, "Date"] + BDay(1)
)

# Remove timezone information from the Date column for easier merging later
news_df["Date"] = news_df["Date"].dt.tz_localize(None)

In [12]:
print(news_df.shape)

print("\nTicker Distribution")
print(news_df["Ticker"].value_counts())

print("\nDate Range")
print(news_df["Date"].min())
print(news_df["Date"].max())

print("\nMissing Values")
print(news_df.isnull().sum())

news_df.head()

(5511, 4)

Ticker Distribution
Ticker
NVDA     3129
GOOGL    1584
AAPL      469
AMZN      329
Name: count, dtype: int64

Date Range
2011-03-03 00:00:00
2020-06-10 00:00:00

Missing Values
Headline     0
Timestamp    0
Ticker       0
Date         0
dtype: int64


,Headline,Timestamp,Ticker,Date
0,Tech Stocks And FAANGS Strong Again To Start D...,2020-06-10 11:33:00-04:00,AAPL,2020-06-10
1,10 Biggest Price Target Changes For Wednesday,2020-06-10 08:14:00-04:00,AAPL,2020-06-10
2,"Benzinga Pro's Top 5 Stocks To Watch For Wed.,...",2020-06-10 07:53:00-04:00,AAPL,2020-06-10
3,"Deutsche Bank Maintains Buy on Apple, Raises P...",2020-06-10 07:19:00-04:00,AAPL,2020-06-10
4,Apple To Let Users Trade In Their Mac Computer...,2020-06-10 06:27:00-04:00,AAPL,2020-06-10


In [14]:
news_df[
    news_df["Timestamp"].dt.hour >= 16
][["Timestamp", "Date"]].head(20)

,Timestamp,Date
17,2020-06-07 17:03:00-04:00,2020-06-08
18,2020-06-06 18:36:00-04:00,2020-06-08
19,2020-06-05 17:19:00-04:00,2020-06-08
52,2020-05-31 18:19:00-04:00,2020-06-01
58,2020-05-27 19:06:00-04:00,2020-05-28
59,2020-05-27 16:31:00-04:00,2020-05-28
68,2020-05-26 16:35:00-04:00,2020-05-27
85,2020-05-19 16:46:00-04:00,2020-05-20
92,2020-05-16 17:12:00-04:00,2020-05-18
93,2020-05-15 16:25:00-04:00,2020-05-18


In [15]:
# Save the cleaned news dataset
news_df.to_csv("news_clean.csv", index=False)

print("Saved news_clean.csv")
print(news_df.shape)

Saved news_clean.csv
(5511, 4)
